In [2]:
!pip -q install google-generativeai
!apt-get -qq update
!apt-get -qq install -y iverilog

import os, textwrap, json, subprocess, re, sys, time
print("✅ Dependencies installed.")

from google.colab import userdata
import google.generativeai as genai

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Dependencies installed.


In [3]:
# ─── CELL 2: AutoChip core utilities (Gemini version) ────────────────────────
from abc import ABC, abstractmethod

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file
        if self.log_file:
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})
        if self.log_file:
            with open(self.log_file, 'a') as f:
                f.write(f"{role}: {content}\n")

    def get_messages(self):
        return self.messages


class AbstractLLM(ABC):
    @abstractmethod
    def generate(self, conversation: Conversation, num_candidates=1):
        pass


class GeminiLLM(AbstractLLM):
    def __init__(self, model_id="gemini-2.5-flash"):
        self.model_id = model_id
        self.model = genai.GenerativeModel(model_id)

    def generate(self, conversation: Conversation, num_candidates=1):
        parts = []
        for m in conversation.get_messages():
            role = m["role"]
            content = m["content"]
            if role == "system":
                parts.append(f"[SYSTEM]: {content}")
            elif role == "user":
                parts.append(f"[USER]: {content}")
            elif role == "assistant":
                parts.append(f"[ASSISTANT]: {content}")
        full_prompt = "\n\n".join(parts)

        outputs = []
        for _ in range(num_candidates):
            response = self.model.generate_content(full_prompt)
            outputs.append(response.text)
        return outputs


def extract_first_verilog_codeblock(text):
    m = re.search(r"```verilog(.*?)```", text, re.S)
    if m:
        return m.group(1).strip() + "\n"
    m2 = re.search(r"```(.*?)```", text, re.S)
    if m2:
        return m2.group(1).strip() + "\n"
    return text.strip() + "\n"


def write_text(path, s):
    with open(path, "w") as f:
        f.write(s)


def compile_and_sim(outdir, sv_file, tb_file):
    os.makedirs(outdir, exist_ok=True)
    exe_file = os.path.join(outdir, "sim.out")

    compile_cmd = f"iverilog -g2012 -o {exe_file} {sv_file} {tb_file}"
    sim_cmd = f"vvp {exe_file}"

    comp = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
    if comp.returncode != 0:
        return False, compile_cmd, sim_cmd, comp.stdout, comp.stderr, "", ""

    sim = subprocess.run(sim_cmd, shell=True, capture_output=True, text=True)
    ok = (sim.returncode == 0) and (("PASS" in sim.stdout) or ("pass" in sim.stdout))
    return ok, compile_cmd, sim_cmd, comp.stdout, comp.stderr, sim.stdout, sim.stderr


def verilog_loop(design_prompt, out_sv_path, tb_file, max_iterations,
                 model_id="gemini-2.5-flash", num_candidates=3, log="trajectory.log"):

    conv = Conversation(log_file=log)
    conv.add_message("system",
        "You generate synthesizable Verilog. Return ONLY the requested module in a single ```verilog``` code block."
    )
    conv.add_message("user", design_prompt)

    model = GeminiLLM(model_id=model_id)

    for it in range(1, max_iterations + 1):
        print(f"\n🔄 Iteration {it}/{max_iterations}")
        candidates = model.generate(conv, num_candidates=num_candidates)

        for k, txt in enumerate(candidates, start=1):
            parsed = extract_first_verilog_codeblock(txt)
            write_text(out_sv_path, parsed)

            ok, compile_cmd, sim_cmd, c_out, c_err, s_out, s_err = compile_and_sim(
                outdir=os.path.dirname(out_sv_path),
                sv_file=out_sv_path,
                tb_file=tb_file
            )

            print(f"\n--- Candidate {k} ---")
            print("Compile cmd:", compile_cmd)
            if c_err.strip():
                print("Compile stderr:\n", c_err)
            if s_out.strip():
                print("Sim stdout:\n", s_out)

            if ok:
                print(f"✅ PASS on iteration {it}, candidate {k}")
                return parsed

            feedback = "Compilation or simulation failed.\n"
            if c_err.strip():
                feedback += "Compiler error:\n" + c_err + "\n"
            if s_out.strip():
                feedback += "Simulator output:\n" + s_out + "\n"
            if s_err.strip():
                feedback += "Simulator stderr:\n" + s_err + "\n"

            conv.add_message("assistant", txt)
            conv.add_message("user",
                "Fix the Verilog module to satisfy the testbench. "
                "Do NOT write a testbench. "
                "Return ONLY the corrected module in a single ```verilog``` code block.\n\n" + feedback
            )

    raise RuntimeError("AutoChip loop ended without a passing solution.")

print("✅ AutoChip utilities loaded (Gemini 2.5 Flash).")

✅ AutoChip utilities loaded (Gemini 2.5 Flash).


In [4]:
# ─── CELL 3: Setup working directory & download testbench ─────────────────────
ROOT = "/content/example2"
os.makedirs(ROOT, exist_ok=True)
os.chdir(ROOT)

print("✅ Working directory:", os.getcwd())

TB_URL = "https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v"

!curl -L -o sequence_detector_tb.v "$TB_URL"

print("✅ Downloaded testbench:")
print(TB_URL)

print("\n📄 sequence_detector_tb.v")
with open("sequence_detector_tb.v", "r") as f:
    tb_text = f.read()
print(tb_text)

# ─── CELL 4: config.json ──────────────────────────────────────────────────────
config = {
  "general": {
    "name": "sequence_detector",
    "prompt": "prompt.txt",
    "testbench": "sequence_detector_tb.v",
    "iterations": 8,
    "num_candidates": 3,
    "outdir": "out",
    "log": "trajectory.log",
    "model_family": "Google",
    "model_id": "gemini-2.5-flash"
  }
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print("📄 config.json")
print(json.dumps(config, indent=2))

✅ Working directory: /content/example2
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0   5578      0 --:--:-- --:--:-- --:--:--  5577
✅ Downloaded testbench:
https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v

📄 sequence_detector_tb.v
`timescale 1ns/1ps

module tb_sequence_detector();
    reg clk;
    reg reset_n;
    reg [2:0] data;
    wire sequence_found;

    // Instantiate the sequence_detector module
    sequence_detector dut (
        .clk(clk),
        .reset_n(reset_n),
        .data(data),
        .sequence_found(sequence_found)
    );

    // Clock generation
    always begin
        #5 clk = ~clk;
    end

    // Test stimulus task
    task apply_stimulus;
        input [2:0] data_value;
        input integer delay_cycles;
        beg

In [5]:
# ─── CELL 5: Module template & prompt ─────────────────────────────────────────
module_template = textwrap.dedent(r"""
`timescale 1ns/1ps
// sequence_detector: detects 8-step sequence on 3-bit data input
module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,
    input  wire [2:0]  data,
    output reg         sequence_found
);
    // TODO: implement FSM (synthesizable)
endmodule
""").strip() + "\n"

with open("sequence_detector_template.sv", "w") as f:
    f.write(module_template)

print("📄 Initial module template: sequence_detector_template.sv")
print(module_template)

prompt = textwrap.dedent(r"""
You are an autocomplete engine for synthesizable Verilog.

Goal:
Write a synthesizable Verilog FSM that detects the following 8-step sequence
on a 3-bit data input:
  0b001 -> 0b101 -> 0b110 -> 0b000 -> 0b110 -> 0b110 -> 0b011 -> 0b101

The module interface MUST match exactly (copy verbatim):

`timescale 1ns/1ps
module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,
    input  wire [2:0]  data,
    output reg         sequence_found
);

Rules:
- Synthesizable RTL only (no delays #, no $display outside initial).
- Use a single reg [3:0] state register with plain integers 0-7 in case statement.
  Do NOT use typedef enum or localparams.
- ONE sequential always @(posedge clk or negedge reset_n) block for state transitions.
- ONE combinational always @(*) block for sequence_found output only.
- sequence_found must be asserted exactly when state==7 && data==3'b101.
- reset_n is active-low: on (!reset_n) set state <= 0.
- On any mismatch, return to state 0.
- Return ONLY the Verilog module in a single ```verilog``` code block.
- Do NOT output a testbench.

Use/complete this template:

```verilog
`timescale 1ns/1ps
module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,
    input  wire [2:0]  data,
    output reg         sequence_found
);
    // TODO: implement FSM (synthesizable)
endmodule
""").strip() + "\n"

with open("prompt.txt", "w") as f:
    f.write(prompt)

print("📄 prompt.txt")
print(prompt)


📄 Initial module template: sequence_detector_template.sv
`timescale 1ns/1ps
// sequence_detector: detects 8-step sequence on 3-bit data input
module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,
    input  wire [2:0]  data,
    output reg         sequence_found
);
    // TODO: implement FSM (synthesizable)
endmodule

📄 prompt.txt
You are an autocomplete engine for synthesizable Verilog.

Goal:
Write a synthesizable Verilog FSM that detects the following 8-step sequence
on a 3-bit data input:
  0b001 -> 0b101 -> 0b110 -> 0b000 -> 0b110 -> 0b110 -> 0b011 -> 0b101

The module interface MUST match exactly (copy verbatim):

`timescale 1ns/1ps
module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,
    input  wire [2:0]  data,
    output reg         sequence_found
);

Rules:
- Synthesizable RTL only (no delays #, no $display outside initial).
- Use a single reg [3:0] state register with plain integers 0-7 in case statement.
  D

In [6]:
# ─── CELL 6: Run AutoChip loop ────────────────────────────────────────────────
with open("config.json", "r") as f:
    cfg = json.load(f)["general"]

module_base    = cfg["name"]
tb_file        = cfg["testbench"]
iterations     = int(cfg["iterations"])
num_candidates = int(cfg["num_candidates"])
outdir         = cfg["outdir"]
logfile        = cfg["log"]
model_id       = cfg["model_id"]

with open(cfg["prompt"], "r") as f:
    design_prompt = f.read()

os.makedirs(outdir, exist_ok=True)
out_sv_path = os.path.join(outdir, f"{module_base}.sv")

start = time.time()
final_sv = verilog_loop(
    design_prompt=design_prompt,
    out_sv_path=out_sv_path,
    tb_file=tb_file,
    max_iterations=iterations,
    model_id=model_id,
    num_candidates=num_candidates,
    log=logfile
)
elapsed = time.time() - start
print(f"\n✅ Done. Time elapsed: {elapsed:.2f}s")

with open(f"{module_base}_FINAL.sv", "w") as f:
    f.write(final_sv)

print(f"📄 Final RTL saved to: {out_sv_path}")
print(f"📄 Final RTL copied to: {module_base}_FINAL.sv")


🔄 Iteration 1/8

--- Candidate 1 ---
Compile cmd: iverilog -g2012 -o out/sim.out out/sequence_detector.sv sequence_detector_tb.v
Sim stdout:
 All test cases passed.

✅ PASS on iteration 1, candidate 1

✅ Done. Time elapsed: 34.38s
📄 Final RTL saved to: out/sequence_detector.sv
📄 Final RTL copied to: sequence_detector_FINAL.sv


In [7]:
# ─── CELL 7: Final verification ───────────────────────────────────────────────
exe_file    = os.path.join(outdir, "sim_final.out")
compile_cmd = f"iverilog -g2012 -o {exe_file} {out_sv_path} {tb_file}"
sim_cmd     = f"vvp {exe_file}"

print("📋 REQUIRED SIM COMMANDS")
print("iverilog command:\n", compile_cmd)
print("\nvvp command:\n", sim_cmd)

comp = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
print("\n── compile stdout ──\n", comp.stdout)
print("\n── compile stderr ──\n", comp.stderr)
assert comp.returncode == 0, "Compile failed — see stderr above."

sim = subprocess.run(sim_cmd, shell=True, capture_output=True, text=True)
print("\n── sim stdout ──\n", sim.stdout)
print("\n── sim stderr ──\n", sim.stderr)
assert sim.returncode == 0 and (("PASS" in sim.stdout) or ("pass" in sim.stdout)), \
    "Simulation did not PASS."
print("\n✅ PASS evidence captured above.")

📋 REQUIRED SIM COMMANDS
iverilog command:
 iverilog -g2012 -o out/sim_final.out out/sequence_detector.sv sequence_detector_tb.v

vvp command:
 vvp out/sim_final.out

── compile stdout ──
 

── compile stderr ──
 

── sim stdout ──
 All test cases passed.


── sim stderr ──
 

✅ PASS evidence captured above.


In [8]:
# ─── CELL 8: Print full trajectory ───────────────────────────────────────────
print("📋 FULL TRAJECTORY LOG")
with open(logfile, "r") as f:
    traj = f.read()

print(traj[:12000])
if len(traj) > 12000:
    print("\n... (log truncated in display; full log saved to trajectory.log) ...\n")
    print(traj[-4000:])

print("\nKey notes:")
print("- trajectory.log contains system prompt, user prompt, model outputs, and feedback across iterations.")
print(f"- Final passing RTL is: {out_sv_path}")
print(f"- Copy of final RTL is: {module_base}_FINAL.sv")


📋 FULL TRAJECTORY LOG
system: You generate synthesizable Verilog. Return ONLY the requested module in a single ```verilog``` code block.
user: You are an autocomplete engine for synthesizable Verilog.

Goal:
Write a synthesizable Verilog FSM that detects the following 8-step sequence
on a 3-bit data input:
  0b001 -> 0b101 -> 0b110 -> 0b000 -> 0b110 -> 0b110 -> 0b011 -> 0b101

The module interface MUST match exactly (copy verbatim):

`timescale 1ns/1ps
module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,
    input  wire [2:0]  data,
    output reg         sequence_found
);

Rules:
- Synthesizable RTL only (no delays #, no $display outside initial).
- Use a single reg [3:0] state register with plain integers 0-7 in case statement.
  Do NOT use typedef enum or localparams.
- ONE sequential always @(posedge clk or negedge reset_n) block for state transitions.
- ONE combinational always @(*) block for sequence_found output only.
- sequence_found must be as

In [9]:
# ─── CELL 9: Folder listing & commented testbench ─────────────────────────────
print("📁 Folder contents:")
!find /content/example2 -maxdepth 2 -type f -print

with open("sequence_detector_tb.v", "r") as f:
    orig = f.read()

header = """`timescale 1ns/1ps
// NOTE: This testbench is sourced from the ChipChat/AutoChip course materials.
// It instantiates sequence_detector and checks sequence_found for the 8-step sequence.
// We compile and run it with:
//   iverilog -g2012 -o sim.out out/sequence_detector.sv sequence_detector_tb_commented.v
//   vvp sim.out
"""
with open("sequence_detector_tb_commented.v", "w") as f:
    f.write(header + "\n" + orig)

print(orig[:500])









📁 Folder contents:
/content/example2/prompt.txt
/content/example2/sequence_detector_FINAL.sv
/content/example2/config.json
/content/example2/trajectory.log
/content/example2/sequence_detector_template.sv
/content/example2/sequence_detector_tb.v
/content/example2/out/sim.out
/content/example2/out/sim_final.out
/content/example2/out/sequence_detector.sv
`timescale 1ns/1ps

module tb_sequence_detector();
    reg clk;
    reg reset_n;
    reg [2:0] data;
    wire sequence_found;

    // Instantiate the sequence_detector module
    sequence_detector dut (
        .clk(clk),
        .reset_n(reset_n),
        .data(data),
        .sequence_found(sequence_found)
    );

    // Clock generation
    always begin
        #5 clk = ~clk;
    end

    // Test stimulus task
    task apply_stimulus;
        input [2:0] data_value;
        input integer dela
